# ML-05 — Leakage checks

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Feature allowlist

Use explicit allowable signals; reject the response, its source numerator, trends, and identifiers.

In [2]:
d,audit=study.read_data()
assert not set(study.FEATURES)&study.BANNED
print('Target / identifier feature overlap: 0')

Target / identifier feature overlap: 0


## Client and content separation

No client or content appears in multiple folds.

In [3]:
idx,counts=study.split_clients(d)
for a,b in [('train','validation'),('train','test'),('validation','test')]:
 assert set(d.iloc[idx[a]].client_id).isdisjoint(set(d.iloc[idx[b]].client_id))
 assert set(d.iloc[idx[a]].content_id).isdisjoint(set(d.iloc[idx[b]].content_id))
print('All pairwise client/content overlaps: 0')

All pairwise client/content overlaps: 0


## Preprocessing and selection

Imputation/encoding are fit inside training pipelines; method selection uses validation only.

In [4]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print(json.dumps(r['leakage_checks'],indent=2))

{
  "client_overlap": 0,
  "content_overlap": 0,
  "target_or_identifier_features": 0,
  "preprocessing_fit_on_train_only": true,
  "winner_selected_on_validation_only": true,
  "future_forecast_claim": false
}
